<a href="https://colab.research.google.com/github/genstac/Hackathon/blob/main/Machine_Learning_Model_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Import all necessary libraries
import pandas as pd
from google.colab import drive
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

# 1. Mount Google Drive and load the dataset
drive.mount('/content/drive')
file_path = '/content/drive/My Drive/Machine Learning Model/cars_ds_final.csv'
df = pd.read_csv(file_path)

# 2. Data Cleaning and Preparation
df = df.drop(columns=['Unnamed: 0'], axis=1)
df['Ex-Showroom_Price_in_Lakhs'] = pd.to_numeric(df['Ex-Showroom_Price'].str.replace('Rs. ', '', regex=False).str.replace(',', '', regex=False), errors='coerce') / 100000
df['ARAI_Certified_Mileage'] = pd.to_numeric(df['ARAI_Certified_Mileage'].str.replace(' kmpl| km/kg| km/litre', '', regex=True), errors='coerce')
df['Power_HP'] = pd.to_numeric(df['Power'].str.extract(r'(\d+)')[0], errors='coerce')
df['Torque_Nm'] = pd.to_numeric(df['Torque'].str.extract(r'(\d+)')[0], errors='coerce')

# 3. Handle missing values (Updated to remove FutureWarning)
df['ARAI_Certified_Mileage'] = df['ARAI_Certified_Mileage'].fillna(df['ARAI_Certified_Mileage'].median())
df['Power_HP'] = df['Power_HP'].fillna(df['Power_HP'].median())
df['Torque_Nm'] = df['Torque_Nm'].fillna(df['Torque_Nm'].median())
df['Ex-Showroom_Price_in_Lakhs'] = df['Ex-Showroom_Price_in_Lakhs'].fillna(df['Ex-Showroom_Price_in_Lakhs'].median())

# 4. Define features (X) and target (y)
X = df[['ARAI_Certified_Mileage', 'Power_HP', 'Torque_Nm']]
y = df['Ex-Showroom_Price_in_Lakhs']

# 5. Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 6. Train the Linear Regression model
model = LinearRegression()
model.fit(X_train, y_train)

# 7. Get user input from the console
try:
    print("Please enter your desired car specifications:")
    mileage_input = float(input("Enter Mileage (e.g., 20): "))
    hp_input = float(input("Enter Horsepower (HP) (e.g., 120): "))
    torque_input = float(input("Enter Torque (Nm) (e.g., 200): "))

    # Create the user input array. We are explicit with column names to avoid the UserWarning.
    user_input = pd.DataFrame([[mileage_input, hp_input, torque_input]],
                              columns=['ARAI_Certified_Mileage', 'Power_HP', 'Torque_Nm'])

except ValueError:
    print("Invalid input. Please enter numerical values.")
    user_input = None

if user_input is not None:
    # Make a prediction for the user's input
    predicted_price = model.predict(user_input)[0]

    # Find and recommend the best cars from the original dataset (Updated to remove SettingWithCopyWarning)
    df_clean = df.dropna(subset=['Make', 'Model']).copy()
    df_clean.loc[:, 'price_difference'] = np.abs(df_clean['Ex-Showroom_Price_in_Lakhs'] - predicted_price)
    recommended_cars = df_clean.sort_values(by='price_difference').head(5)

    print("\n--- Your Car Recommendation ---")
    print(f"Based on your input, the predicted price is: Rs. {predicted_price * 100000:.2f}")
    print("\nWe recommend the following cars based on their price and features:")

    for index, row in recommended_cars.iterrows():
        print("-------------------")
        print(f"Make: {row['Make']}")
        print(f"Model: {row['Model']}")
        print(f"Variant: {row['Variant']}")
        print(f"Actual Ex-Showroom Price: {row['Ex-Showroom_Price']}")
        print(f"Distance to Predicted Price: Rs. {row['price_difference'] * 100000:.2f}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Please enter your desired car specifications:
Enter Mileage (e.g., 20): 20
Enter Horsepower (HP) (e.g., 120): 120
Enter Torque (Nm) (e.g., 200): 200

--- Your Car Recommendation ---
Based on your input, the predicted price is: Rs. 2103656.49

We recommend the following cars based on their price and features:
-------------------
Make: Toyota
Model: Innova Crysta
Variant: 2.7 Zx At 7 Str
Actual Ex-Showroom Price: Rs. 21,03,000
Distance to Predicted Price: Rs. 656.49
-------------------
Make: Toyota
Model: Innova Crysta
Variant: Touring Sport 2.4 Vx 7 Str
Actual Ex-Showroom Price: Rs. 20,97,000
Distance to Predicted Price: Rs. 6656.49
-------------------
Make: Toyota
Model: Innova Crysta
Variant: 2.4 Zx 7 Str
Actual Ex-Showroom Price: Rs. 21,13,000
Distance to Predicted Price: Rs. 9343.51
-------------------
Make: Mg
Model: Zs Ev
Variant: Excite
Actual Ex-Showro

In [ ]:
%%writefile car_recommender_app.py
import pandas as pd
from google.colab import drive
import numpy as np
import streamlit as st
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# --- Configuration ---
# Set the page configuration for a modern, wide-screen feel
st.set_page_config(layout="wide", page_title="Car Recommendation System")

# --- Constants & Data Path ---
# CRITICAL: This path points to the CSV file you uploaded to your Google Drive
FILE_PATH = '/content/drive/My Drive/Machine Learning Model/car_ds_final.csv'


# --- Core Data Processing and Model Training Function ---
# This function is decorated with @st.cache_resource, so it runs only once, making the app fast.

@st.cache_resource
def load_and_train_model():
    """
    Handles mounting, data loading, cleaning, imputation, and model training.
    Returns the cleaned DataFrame, the trained model, and the R-squared score.
    """
    try:
        # Mount the drive inside the function for seamless execution
        print("Attempting to mount Google Drive...")
        drive.mount('/content/drive', force_remount=True)
        df = pd.read_csv(FILE_PATH)
        print("Data loaded successfully.")
    except FileNotFoundError:
        st.error(f"Error: The file at '{FILE_PATH}' was not found. Please ensure your drive is mounted and the path is correct.")
        return None, None, 0
    except Exception as e:
        st.error(f"An unexpected error occurred during data loading: {e}")
        return None, None, 0

    # 1. Initial Cleaning
    df = df.drop(columns=['Unnamed: 0'], axis=1, errors='ignore')

    # 2. Clean Target: Ex-Showroom_Price_in_Lakhs
    df['Ex-Showroom_Price_in_Lakhs'] = pd.to_numeric(
        df['Ex-Showroom_Price'].str.replace('Rs. ', '', regex=False).str.replace(',', '', regex=False),
        errors='coerce'
    ) / 100000

    # 3. Clean Features
    df['ARAI_Certified_Mileage'] = pd.to_numeric(
        df['ARAI_Certified_Mileage'].str.replace(' kmpl| km/kg| km/litre', '', regex=True),
        errors='coerce'
    )
    # Using explicit series extraction to avoid warnings
    power_series = df['Power'].str.extract(r'(\d+)')[0]
    torque_series = df['Torque'].str.extract(r'(\d+)')[0]
    df['Power_HP'] = pd.to_numeric(power_series, errors='coerce')
    df['Torque_Nm'] = pd.to_numeric(torque_series, errors='coerce')

    # 4. Imputation (Filling NaNs with Median - updated for clean execution)
    df['ARAI_Certified_Mileage'] = df['ARAI_Certified_Mileage'].fillna(df['ARAI_Certified_Mileage'].median())
    df['Power_HP'] = df['Power_HP'].fillna(df['Power_HP'].median())
    df['Torque_Nm'] = df['Torque_Nm'].fillna(df['Torque_Nm'].median())
    df['Ex-Showroom_Price_in_Lakhs'] = df['Ex-Showroom_Price_in_Lakhs'].fillna(df['Ex-Showroom_Price_in_Lakhs'].median())

    # 5. Define features (X) and target (y)
    X = df[['ARAI_Certified_Mileage', 'Power_HP', 'Torque_Nm']]
    y = df['Ex-Showroom_Price_in_Lakhs']

    # 6. Split and Train
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    model = LinearRegression()
    model.fit(X_train, y_train)

    # Calculate R-squared for model evaluation
    y_pred = model.predict(X_test)
    r2 = r2_score(y_test, y_pred)

    return df, model, r2

# --- Recommendation Logic ---

def get_recommendations(df, model, mileage, hp, torque, top_n=5):
    """
    Predicts price and finds the top N closest cars based on price deviation.
    """
    # Create user input DataFrame matching the format expected by the model
    user_input = pd.DataFrame([[mileage, hp, torque]],
                              columns=['ARAI_Certified_Mileage', 'Power_HP', 'Torque_Nm'])

    # Predict the price (in Lakhs)
    predicted_price = model.predict(user_input)[0]

    # Clean the df for display purposes (avoid recommending cars with no descriptive data)
    df_clean = df.dropna(subset=['Make', 'Model', 'Variant', 'Ex-Showroom_Price']).copy()

    # Calculate the absolute difference from the predicted price
    df_clean.loc[:, 'price_difference'] = np.abs(df_clean['Ex-Showroom_Price_in_Lakhs'] - predicted_price)

    # Sort and select the top N recommendations
    recommended_cars = df_clean.sort_values(by='price_difference').head(top_n)

    # Format the output for the Streamlit table
    results = recommended_cars[[
        'Make', 'Model', 'Variant', 'Ex-Showroom_Price', 'ARAI_Certified_Mileage', 'Power_HP', 'price_difference'
    ]].rename(columns={
        'ARAI_Certified_Mileage': 'Mileage (kmpl/kg)',
        'Power_HP': 'Power (HP)',
        'price_difference': 'Price Deviation (Lakhs)'
    })

    # Format Deviation into full Rupees for readability
    results.loc[:, 'Price Deviation (Lakhs)'] = (results['Price Deviation (Lakhs)'] * 100000).round(2).apply(
        lambda x: f"₹{x:,.0f}"
    )

    formatted_predicted_price = f"₹{(predicted_price * 100000):,.0f}"

    return results, formatted_predicted_price, predicted_price

# --- Streamlit Application Layout ---

def app():
    st.title("🇮🇳 Machine Learning Car Recommender")
    st.markdown("---")

    df, model, r2_score = load_and_train_model()

    if df is None:
        return

    # Display Model Performance Sidebar
    with st.sidebar:
        st.header("Project Insights")
        st.metric(label="Model Used", value="Linear Regression")
        st.metric(label="R-squared Score (Accuracy)", value=f"{r2_score:.2f}")
        st.markdown(f"*(Model explains {r2_score*100:.0f}% of price variation)*")
        st.markdown("---")
        st.caption("Adjust the sliders below to get started.")

    # --- User Input Sliders ---
    st.header("🎯 Your Desired Car Specifications")

    col1, col2, col3 = st.columns(3)

    # Calculate min/max/median for sliders to provide sensible defaults
    min_mileage, max_mileage = df['ARAI_Certified_Mileage'].min(), df['ARAI_Certified_Mileage'].max()
    median_mileage = df['ARAI_Certified_Mileage'].median()

    min_hp, max_hp = df['Power_HP'].min(), df['Power_HP'].max()
    median_hp = df['Power_HP'].median()

    min_torque, max_torque = df['Torque_Nm'].min(), df['Torque_Nm'].max()
    median_torque = df['Torque_Nm'].median()

    with col1:
        mileage_input = st.slider(
            "Certified Mileage (kmpl/km/kg)",
            min_value=float(min_mileage),
            max_value=float(max_mileage),
            value=float(median_mileage),
            step=0.1,
            format="%.1f"
        )

    with col2:
        hp_input = st.slider(
            "Horsepower (HP)",
            min_value=float(min_hp),
            max_value=float(max_hp),
            value=float(median_hp),
            step=1.0,
            format="%.0f"
        )

    with col3:
        torque_input = st.slider(
            "Torque (Nm)",
            min_value=float(min_torque),
            max_value=float(max_torque),
            value=float(median_torque),
            step=1.0,
            format="%.0f"
        )

    st.markdown("---")

    # --- Generate Recommendations ---

    results_df, predicted_price_str, predicted_price_lakhs = get_recommendations(
        df, model, mileage_input, hp_input, torque_input
    )

    st.header("✨ Top 5 Recommended Cars")
    st.success(f"Predicted Ex-Showroom Price for your specifications: **{predicted_price_str}**")

    # Display results table
    st.dataframe(results_df, use_container_width=True)

    # --- Conclusion for the Project ---
    st.markdown("---")
    st.subheader("B.Tech Project Conclusion & Future Scope")

    st.markdown(
        """
        This is a robust minimum viable product (MVP) for an ML-powered car recommendation engine.
        It proves the concept of using regression to guide content-based filtering.

        **Future Scope for Project Expansion:**
        1.  **Advanced Modeling:** Implement a **Random Forest Regressor** to capture non-linear feature relationships, which will significantly increase the R-squared score.
        2.  **Feature Enrichment:** Use **One-Hot Encoding** to incorporate crucial categorical variables like **`Fuel_Type`** and **`Body_Type`** into the model for deeper predictive insights.
        3.  **Visualization:** Add interactive charts (e.g., Plotly or Altair) to the app to show how the user's input compares to the rest of the dataset.
        """
    )


if __name__ == '__main__':
    app()

Writing car_recommender_app.py


In [ ]:
# Launch the Streamlit app directly inside the Colab output cell
from google.colab import output
import subprocess
import time

print("Starting Streamlit app directly in Colab output...")

# Run Streamlit in the background
# We run it using the Python interpreter (python -m) for maximum compatibility.
subprocess.Popen(["python", "-m", "streamlit", "run", "car_recommender_app.py", "--server.port", "8501", "--server.enableCORS", "false"])

# Wait briefly for Streamlit to initialize
time.sleep(5)

# Display the Streamlit app in an output frame (this replaces the ngrok step)
output.serve_kernel_port_as_iframe(8501, height=800)

Starting Streamlit app directly in Colab output...


<IPython.core.display.Javascript object>